In [ ]:
import os
from copy import deepcopy
from typing import Dict

import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
from sklearn.metrics import r2_score
from torch_geometric.data import Data
from tqdm import tqdm

import barostat_parameters
from graph_utils import prepare_traj
from itpo_weights import DatasetType, ITPOWeights, ModelType
from simulator_model import Model as VelocityModel
from training_utils import freeze_normalizer
from utils import (
    build_velocity_graph_correction,
    calc_p_ratio_box_tensor,
    load_and_split_dataset,
    specialized_rollout,
)


### Data

We search for ITPO weights using intermidiate ($\nu \in (0.1, 0.2)$) data.

To check the how well these weights perform, different validation/test is used.

In [ ]:
poisson_buckets = [
    {"max": 0.1, "count": 100},                # P < 0.1
    {"min": 0.1, "max": 0.2, "count": 100},    # 0.1 <= P < 0.2
    {"min": 0.2, "count": 200}                 # P >= 0.2
]

dataset_type = DatasetType.NodeOptimized

train_files, val_files, test_files = load_and_split_dataset(
    registry_path="./data/data_registry.csv",
    target_data_type=dataset_type,
    possion_buckets=poisson_buckets,
    split_ratios=(0.7, 0.15, 0.15),
    seed=42
)

# Load actual data
data = {
    'train': {},
    'val' : {},
    'test' : {},
}

print("Loading data...")
for key in data.keys():
    if key == 'train':
        data[key] = [torch.load(file, weights_only=False) for file in tqdm(train_files, desc=f"{key:<5} data")]
    elif key == 'val':
        data[key] = [torch.load(file, weights_only=False) for file in tqdm(val_files, desc=f"{key:<5} data")]
    elif key == 'test':
        data[key] = [torch.load(file, weights_only=False) for file in tqdm(test_files, desc=f"{key:<5} data")]
    else:
        raise ValueError(f"Unexpected key in data dictionary: {key}. ")

print("\nPreparing data...")
for data_type, sims in data.items():
    prepared = []
    for sim in tqdm(sims, desc=f"{data_type:<5} data"):
        prepared_sim = prepare_traj(sim, calc_angles=False)
        prepared.append(prepared_sim)
    data[data_type] = prepared

print(f"\nTrain data: {len(data['train'])} sims.")
print(f"Val data:   {len(data['val'])} sims.")
print(f"Test data:  {len(data['test'])} sims.")

#### Show $\nu$ distribution

In [ ]:
def visualize_nu_disribution(data: Dict):
    ps = {}
    all_values = []

    for data_type, sims in data.items():
        ps[data_type] = [calc_p_ratio_box_tensor(sim).item() for sim in sims]
        all_values.extend(ps[data_type])

    min_val = min(all_values)
    max_val = max(all_values)
    common_bins = np.linspace(min_val, max_val, 30) 

    for data_type, values in ps.items():
        plt.hist(
        values, 
        bins=common_bins, 
        edgecolor='black', 
        alpha=0.6, 
        label=f"{data_type} data"
    )

    plt.title("$\\nu$ distribution")
    plt.xlabel("GT LAMMPS $\\nu$")
    plt.ylabel("N")
    plt.legend()
    plt.show()

visualize_nu_disribution(data)


### Load pretrained models

In [ ]:
simulator_type = ModelType.SimulatorCascade # ModelType.GNNModel | ModelType.SimulatorCascade

match simulator_type:
    case ModelType.GNNModel:
        mp_layers = 2
        mlp = 3
        hidden_size = 128
        history = 3
        device = "cuda"

        init_graph = Data(x=torch.ones((100, history*2)), edge_attr=torch.ones((100, 4)))

        models = {}

        model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
        model.load_checkpoint(f"./trained_models/{dataset_type}/OST/model_P>0.2_h{history}_nl{mp_layers}_mlp{mlp}_epochs100.pt")
        model = freeze_normalizer(model)
        models[f"h{history} P>0.2 ost"] = model

        model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
        model.load_checkpoint(f"./trained_models/{dataset_type}/OST/model_P>0.1_h{history}_nl{mp_layers}_mlp{mlp}_epochs100.pt")
        model = freeze_normalizer(model)
        models[f"h{history} P>0.1 ost"] = model

        model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
        model.load_checkpoint(f"./trained_models/{dataset_type}/MST/model_P>0.2_h{history}_nl{mp_layers}_mlp{mlp}_epochs100.pt")
        model = freeze_normalizer(model)
        models[f"h{history} P>0.2 mst"] = model

        model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
        model.load_checkpoint(f"./trained_models/{dataset_type}/MST/model_P>0.1_h{history}_nl{mp_layers}_mlp{mlp}_epochs100.pt")
        model = freeze_normalizer(model)
        models[f"h{history} P>0.1 mst"] = model

        for i, model_name in enumerate(models.keys()):
            print(f"{i+1}. Model {model_name}.")

    case ModelType.SimulatorCascade:
        hidden_size = 128
        mp_layers = 2
        mlp = 3
        epochs = 100

        device = 'cuda'
        model_save_path = os.path.join("./trained_models", f"{dataset_type}", "cascade", "refined_P>0.2")
        n_models = len(os.listdir(model_save_path))

        models = []
        for h in tqdm(range(n_models)):
            if h == 0:
                n_graphs = 1
            else:
                n_graphs = h+1
            
            init_graph = build_velocity_graph_correction([data['val'][0][i].cpu().detach() for i in range(n_graphs)], panic_at_positions=False).to(device)

            current_h_model = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
            current_h_model.load_checkpoint(os.path.join(model_save_path, f"model_refined_h{h}_nl{mp_layers}_mlp{mlp}_epochs{epochs}.pt"))
            current_h_model = freeze_normalizer(current_h_model)
            for param in current_h_model.parameters():
                param.requires_grad = False
            models.append(current_h_model)

        print(f"Loaded cascade of {len(models)} pretrained models.")    


### Find best ITPO weights for GNN simulator model

In [ ]:
models.keys()

In [ ]:
model = models['h3 P>0.2 ost']

num_rollout_steps = 50
dump_period = 200

barostat_config = barostat_parameters.node_optimizated if dataset_type == DatasetType.NodeOptimized else barostat_parameters.stiff_optimized

def objective(trial: optuna.Trial) -> float:
    
    # Define hyperparameter search space
    refinement_iterations = trial.suggest_int("refinement_iterations", 5, 30)
    lr = trial.suggest_float("lr", 1e-8, 1e-3, log=True)
    lambda_force = trial.suggest_float("lambda_force", 1e-8, 1.0, log=True)
    lambda_energy = trial.suggest_float("lambda_energy", 1e-8, 1.0, log=True)
    lambda_pressure = trial.suggest_float("lambda_pressure", 1e-8, 1.0, log=True)

    # Collect results here
    gt_poissons = []
    pred_poissons = []

    # Evaluation Loop
    intermidiate_data = [sim for sim in data['train'] if calc_p_ratio_box_tensor(sim) >= 0.1 and calc_p_ratio_box_tensor(sim) < 0.2]
    for sim in intermidiate_data[:5]:

        fresh_clone_graph = deepcopy(sim[0].cpu().detach().clone())
        
        rollout = specialized_rollout(
            starting_graph=fresh_clone_graph,
            gnn_simulator=model,
            gnn_history=history,
            barostat_config=barostat_config,
            itpo_weights=ITPOWeights(refinement_iterations, lr, lambda_force, lambda_energy, lambda_pressure),
            md_steps=(dump_period*history)+1,
            rollout_steps=num_rollout_steps,
            device="cuda"
        )
        
        pred_p = calc_p_ratio_box_tensor(rollout)
        
        # Check for NaNs
        if torch.isnan(pred_p) or torch.isinf(pred_p):
            raise ValueError(f"Physics Divergence (NaN/Inf detected): {pred_p}")

        pred_poissons.append(pred_p.item())
        
        gt_p = calc_p_ratio_box_tensor(sim[:num_rollout_steps])
        gt_poissons.append(gt_p.item())

    # Calculate Objective (Maximize R^2)        
    score = r2_score(gt_poissons, pred_poissons)
    
    return score

# Create the study. We want to maximize the R2 score.
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner()
)

print("Starting Optuna optimization...")
study.optimize(objective, n_trials=300, show_progress_bar=False)

print("Best Trial:")
trial = study.best_trial
print(f"  R2 Score: {trial.value}")
print("  Best Hyperparameters:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")


### Find ITPO weights for simulator cascade

In [ ]:
from utils import specialized_rollout_cascade

intermidiate_data = [sim for sim in data['train'] if calc_p_ratio_box_tensor(sim) >= 0.1 and calc_p_ratio_box_tensor(sim) < 0.2]
factors = [(sim[4].box_tensor[0]/sim[3].box_tensor[0]).item() for sim in intermidiate_data]
mean_factor = sum(factors)/len(factors)

num_rollout_steps = 50
dump_period = 200

barostat_config = barostat_parameters.node_optimizated if dataset_type == DatasetType.NodeOptimized else barostat_parameters.stiff_optimized

def objective(trial: optuna.Trial) -> float:

    # Define hyperparameter search space
    refinement_iterations = trial.suggest_int("refinement_iterations", 5, 50)
    lr = trial.suggest_float("lr", 1e-8, 1e-3, log=True)
    lambda_force = trial.suggest_float("lambda_force", 1e-8, 1.0, log=True)
    lambda_energy = trial.suggest_float("lambda_energy", 1e-8, 1.0, log=True)
    lambda_pressure = trial.suggest_float("lambda_pressure", 1e-8, 1.0, log=True)

    # Collect results
    gt_poissons = []
    pred_poissons = []
    
    # Evaluation Loop
    for sim in intermidiate_data[:5]:
        fresh_clone_graph = deepcopy(sim[0].cpu().detach().clone())
        input_graphs = [fresh_clone_graph, ]
        rollout = specialized_rollout_cascade(
            starting_graph=input_graphs[0],
            gnn_models=models,
            barostat_config=barostat_config,
            box_compression_factor=mean_factor,
            itpo_weights=ITPOWeights(refinement_iterations, lr, lambda_force, lambda_energy, lambda_pressure),
            rollout_steps=num_rollout_steps,
            device="cuda",
        )


        pred_p = calc_p_ratio_box_tensor(rollout)
        
        # Check for NaNs (physics explosion)
        if torch.isnan(pred_p) or torch.isinf(pred_p):
            raise ValueError("Physics Divergence (NaN/Inf detected)")

        pred_poissons.append(pred_p.item())
        
        gt_p = calc_p_ratio_box_tensor(sim[:num_rollout_steps])
        gt_poissons.append(gt_p.item())

    if len(pred_poissons) < 2:
        return -100.0 # Safety check for R2 calculation
        
    score = r2_score(gt_poissons, pred_poissons)
    
    return score


study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner()
)

print("Starting Optuna optimization...")
study.optimize(objective, n_trials=300, show_progress_bar=False)

print("\nOptimization Finished!")
print("Best Trial:")
trial = study.best_trial
print(f"  R2 Score: {trial.value}")
print("  Best Hyperparameters:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")
